# Topic Modeling LDA, FTC Clustering, dan Rule-Based NER - Full Notebook

Alur proyek: preprocessing teks, LDA topic modeling, clustering dokumen menggunakan metode FTC berbasis frequent term dan Entropy Overlap, lalu Named Entity Recognition dengan rule-based NER.


In [2]:
from __future__ import annotations

import math
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 160)


In [3]:
DATA_PATH = Path("komentar_youtube.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TEXT_COLUMN_CANDIDATES = ["komentar", "text", "comment", "comments", "Komentar", "Text"]
N_TOPICS = 5
N_CLUSTERS = 4
LDA_ITER = 300
MIN_DF = 2
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
TEXT_COL = next((col for col in TEXT_COLUMN_CANDIDATES if col in df.columns), None)

if TEXT_COL is None:
    raise ValueError(
        f"Kolom teks tidak ditemukan. Kolom yang tersedia: {df.columns.tolist()}"
    )

if "comment_id" not in df.columns:
    df.insert(0, "comment_id", range(1, len(df) + 1))

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

print("Dataset:", DATA_PATH)
print("Shape:", df.shape)
print("Kolom teks:", TEXT_COL)
print("Kolom:", df.columns.tolist())
df[["comment_id", TEXT_COL]].head(10)


Dataset: komentar_youtube.csv
Shape: (500, 2)
Kolom teks: komentar
Kolom: ['comment_id', 'komentar']


,comment_id,komentar
0,1,"Org sepintar ini,kalah saya tukang mebel.😢😢😢"
1,2,Perlawanan ojol dgn tolak jokodok
2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya"
3,4,Kejahatan Luarbiasa DiNegeri ini
4,5,Negara Konoha memang negara paling aneh sedunia..
5,6,"Yg membuka lapangan kerja ber juta orang di Indonesia di bui,sedangkan koruptor di Indonesia yg merajalela,semakin solid,(pasti bung Karno&Soeharto)sedih me..."
6,7,Biangkeroknya tangkap! Tapi siapa yang bisa nangkap???😢 semoga Tuhan memberikan hidajah untuk orang2 jujur. Aamiin
7,8,Yg Korupsi MBG tuh...di Hukum Mati
8,9,Kenapa begitu.. coba dehh tolong menolong logikanya egois .. yang menolong jangan setengah-setengah..
9,10,Astaghfirullah


## 1. Preprocessing Teks

Tahap ini membersihkan URL, mention, hashtag, simbol, angka, slang sederhana, menghapus stopword bahasa Indonesia dengan **Sastrawi**, lalu melakukan stemming dengan **Sastrawi Stemmer**.

In [4]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword_factory = StopWordRemoverFactory()
stemmer_factory = StemmerFactory()

SASTRAWI_STOPWORDS = set(stopword_factory.get_stop_words())
CUSTOM_STOPWORDS_ID = {
    "aja", "banget", "deh", "dong", "gak", "ga", "nggak", "nya", "nih",
    "sih", "ya", "yg", "youtube", "komen", "komentar", "wkwk", "wk",
}
STOPWORDS_ID = SASTRAWI_STOPWORDS | CUSTOM_STOPWORDS_ID
stemmer = stemmer_factory.create_stemmer()

SLANG_MAP = {
    "aja": "saja",
    "anjg": "anjing",
    "bgt": "banget",
    "bgtt": "banget",
    "dgn": "dengan",
    "dlm": "dalam",
    "dr": "dari",
    "emng": "memang",
    "ga": "tidak",
    "gak": "tidak",
    "gk": "tidak",
    "jd": "jadi",
    "jdi": "jadi",
    "jkwi": "jokowi",
    "karna": "karena",
    "krn": "karena",
    "msh": "masih",
    "ngak": "tidak",
    "ngga": "tidak",
    "nggak": "tidak",
    "org": "orang",
    "pd": "pada",
    "ri": "indonesia",
    "tdk": "tidak",
    "tp": "tapi",
    "utk": "untuk",
    "yg": "yang",
    "tau": "tahu",
    "sdh": "sudah",
    "bpk": "bapak"
}

RE_URL = re.compile(r"https?://\S+|www\.\S+")
RE_MENTION = re.compile(r"@\w+")
RE_HASHTAG = re.compile(r"#(\w+)")
RE_ALPHA = re.compile(r"[^a-zA-Z\s]")
RE_SPACES = re.compile(r"\s+")
RE_REPEAT = re.compile(r"(.)\1{2,}")
RE_LAUGH = re.compile(r"^(ha)+h?$|^(wkwk)+$|^wk+$|^xixi+$")


In [5]:
def normalize_text(text: object) -> str:
    value = "" if pd.isna(text) else str(text)
    value = value.lower()
    value = RE_URL.sub(" ", value)
    value = RE_MENTION.sub(" ", value)
    value = RE_HASHTAG.sub(r"\1", value)
    value = RE_ALPHA.sub(" ", value)
    value = RE_SPACES.sub(" ", value).strip()
    return value


def stem_token(token: str) -> str:
    return stemmer.stem(token)


def tokenize(text: object) -> list[str]:
    normalized = normalize_text(text)
    tokens = []

    for token in normalized.split():
        token = RE_REPEAT.sub(r"\1\1", token)
        token = SLANG_MAP.get(token, token)

        if len(token) < 3:
            continue
        if token in STOPWORDS_ID:
            continue
        if RE_LAUGH.match(token):
            continue

        token = stem_token(token)

        if len(token) < 3:
            continue
        if token in STOPWORDS_ID:
            continue

        tokens.append(token)

    return tokens


df["normalized_text"] = df[TEXT_COL].apply(normalize_text)
df["tokens"] = df[TEXT_COL].apply(tokenize)
df["clean_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))
df["token_count"] = df["tokens"].apply(len)

print("Jumlah stopword Sastrawi + custom:", len(STOPWORDS_ID))
print("Stemmer: Sastrawi")
df[[TEXT_COL, "normalized_text", "clean_text", "token_count"]].head(10)


Jumlah stopword Sastrawi + custom: 138
Stemmer: Sastrawi


,komentar,normalized_text,clean_text,token_count
0,"Org sepintar ini,kalah saya tukang mebel.😢😢😢",org sepintar ini kalah saya tukang mebel,orang pintar kalah tukang mebel,5
1,Perlawanan ojol dgn tolak jokodok,perlawanan ojol dgn tolak jokodok,lawan ojol tolak jokodok,4
2,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",pilpres yg untuk menentukan nasib juta orang indonesia saja bisa diatur apalagi cuma untuk orang indonesia ya gak pak nadiem sabar lah ya,pilpres nasib juta orang indonesia atur cuma orang indonesia pak nadiem sabar lah,13
3,Kejahatan Luarbiasa DiNegeri ini,kejahatan luarbiasa dinegeri ini,jahat luarbiasa negeri,3
4,Negara Konoha memang negara paling aneh sedunia..,negara konoha memang negara paling aneh sedunia,negara konoha memang negara paling aneh dunia,7
5,"Yg membuka lapangan kerja ber juta orang di Indonesia di bui,sedangkan koruptor di Indonesia yg merajalela,semakin solid,(pasti bung Karno&Soeharto)sedih me...",yg membuka lapangan kerja ber juta orang di indonesia di bui sedangkan koruptor di indonesia yg merajalela semakin solid pasti bung karno soeharto sedih mel...,buka lapang kerja ber juta orang indonesia bui koruptor indonesia rajalela makin solid bung karno soeharto sedih lihat rakyat indonesia sengsara,21
6,Biangkeroknya tangkap! Tapi siapa yang bisa nangkap???😢 semoga Tuhan memberikan hidajah untuk orang2 jujur. Aamiin,biangkeroknya tangkap tapi siapa yang bisa nangkap semoga tuhan memberikan hidajah untuk orang jujur aamiin,biangkeroknya tangkap siapa nangkap moga tuhan beri hidajah orang jujur aamiin,11
7,Yg Korupsi MBG tuh...di Hukum Mati,yg korupsi mbg tuh di hukum mati,korupsi mbg tuh hukum mati,5
8,Kenapa begitu.. coba dehh tolong menolong logikanya egois .. yang menolong jangan setengah-setengah..,kenapa begitu coba dehh tolong menolong logikanya egois yang menolong jangan setengah setengah,coba dehh logika egois jangan tengah tengah,7
9,Astaghfirullah,astaghfirullah,astaghfirullah,1


## 2. LDA Topic Modeling

LDA di bawah memakai collapsed Gibbs sampling sederhana berbasis `numpy`, sehingga notebook tetap bisa berjalan tanpa `sklearn` atau `gensim`.

In [6]:
def build_vocabulary(
    token_lists: Sequence[Sequence[str]],
    min_df: int = 2,
    max_df_ratio: float = 0.8,
    max_features: int = 3000,
) -> dict[str, int]:
    doc_freq = Counter()
    term_freq = Counter()

    for tokens in token_lists:
        term_freq.update(tokens)
        doc_freq.update(set(tokens))

    doc_count = max(1, len(token_lists))
    max_df = max(1, math.ceil(doc_count * max_df_ratio))

    terms = [term for term, df_value in doc_freq.items() if min_df <= df_value <= max_df]
    terms.sort(key=lambda term: (-term_freq[term], term))
    terms = terms[:max_features]

    return {term: index for index, term in enumerate(terms)}


def docs_to_word_ids(
    token_lists: Sequence[Sequence[str]],
    vocabulary: dict[str, int],
) -> list[list[int]]:
    return [[vocabulary[token] for token in tokens if token in vocabulary] for tokens in token_lists]


@dataclass
class SimpleLDA:
    n_topics: int = 5
    n_iter: int = 300
    alpha: float = 0.1
    beta: float = 0.01
    random_state: int = 42

    def fit(self, documents: Sequence[Sequence[int]], vocab_size: int) -> "SimpleLDA":
        if vocab_size == 0:
            raise ValueError("Vocabulary kosong. Turunkan MIN_DF atau cek isi kolom text.")
        if self.n_topics < 2:
            raise ValueError("N_TOPICS minimal 2.")

        rng = np.random.default_rng(self.random_state)
        doc_count = len(documents)

        self.doc_topic_counts_ = np.zeros((doc_count, self.n_topics), dtype=np.int64)
        self.topic_word_counts_ = np.zeros((self.n_topics, vocab_size), dtype=np.int64)
        self.topic_counts_ = np.zeros(self.n_topics, dtype=np.int64)
        self.assignments_ = []

        for doc_id, words in enumerate(documents):
            topics = rng.integers(0, self.n_topics, size=len(words), endpoint=False)
            self.assignments_.append(topics)

            for word_id, topic_id in zip(words, topics):
                self.doc_topic_counts_[doc_id, topic_id] += 1
                self.topic_word_counts_[topic_id, word_id] += 1
                self.topic_counts_[topic_id] += 1

        for _ in range(self.n_iter):
            for doc_id, words in enumerate(documents):
                topics = self.assignments_[doc_id]

                for token_index, word_id in enumerate(words):
                    old_topic = topics[token_index]
                    self.doc_topic_counts_[doc_id, old_topic] -= 1
                    self.topic_word_counts_[old_topic, word_id] -= 1
                    self.topic_counts_[old_topic] -= 1

                    topic_prob = (
                        (self.doc_topic_counts_[doc_id] + self.alpha)
                        * (self.topic_word_counts_[:, word_id] + self.beta)
                        / (self.topic_counts_ + vocab_size * self.beta)
                    )
                    topic_prob = topic_prob / topic_prob.sum()

                    new_topic = rng.choice(self.n_topics, p=topic_prob)
                    topics[token_index] = new_topic
                    self.doc_topic_counts_[doc_id, new_topic] += 1
                    self.topic_word_counts_[new_topic, word_id] += 1
                    self.topic_counts_[new_topic] += 1

        theta = self.doc_topic_counts_ + self.alpha
        self.doc_topic_ = theta / theta.sum(axis=1, keepdims=True)

        phi = self.topic_word_counts_ + self.beta
        self.topic_word_ = phi / phi.sum(axis=1, keepdims=True)

        return self

    def top_words(self, id_to_word: Sequence[str], n_words: int = 10) -> pd.DataFrame:
        rows = []

        for topic_id in range(self.n_topics):
            word_order = np.argsort(-self.topic_word_[topic_id])[:n_words]
            for rank, word_id in enumerate(word_order, start=1):
                rows.append(
                    {
                        "topic_id": topic_id,
                        "rank": rank,
                        "term": id_to_word[word_id],
                        "probability": float(self.topic_word_[topic_id, word_id]),
                    }
                )

        return pd.DataFrame(rows)


In [20]:
token_lists = df["tokens"].tolist()
vocabulary = build_vocabulary(token_lists, min_df=MIN_DF)

if len(vocabulary) < max(10, N_TOPICS * 2):
    vocabulary = build_vocabulary(token_lists, min_df=1)

id_to_word = [None] * len(vocabulary)
for term, word_id in vocabulary.items():
    id_to_word[word_id] = term

documents = docs_to_word_ids(token_lists, vocabulary)

lda_model = SimpleLDA(
    n_topics=N_TOPICS,
    n_iter=LDA_ITER,
    random_state=RANDOM_STATE,
).fit(documents, vocab_size=len(vocabulary))

topic_words = lda_model.top_words(id_to_word, n_words=12)
topic_matrix = pd.DataFrame(
    lda_model.doc_topic_,
    columns=[f"topic_{topic_id}" for topic_id in range(N_TOPICS)],
)

print("Jumlah komentar:", len(df))
print("Ukuran vocabulary:", len(vocabulary))
lda_table = topic_words.copy()

lda_table["probability"] = lda_table["probability"].round(4)

lda_table


Jumlah komentar: 500
Ukuran vocabulary: 528


,topic_id,rank,term,probability
0,0,1,pak,0.0571
1,0,2,allah,0.0444
2,0,3,nadiem,0.0381
3,0,4,adil,0.0338
4,0,5,moga,0.0338
5,0,6,siapa,0.0285
6,0,7,hakim,0.0254
7,0,8,tuhan,0.0233
8,0,9,orang,0.0222
9,0,10,bapak,0.0211


## 3. Clustering Komentar dengan FTC

Clustering menggunakan **Frequent Term Based Clustering (FTC)**. Kandidat cluster dibentuk dari frequent term/term-set, lalu dipilih iteratif memakai nilai **Entropy Overlap (EO)** minimum.


In [8]:
from itertools import combinations

FTC_MIN_SUPPORT = 8
FTC_MAX_K = 2
FTC_MAX_TERMS = 120


def generate_frequent_termsets(
    token_lists: Sequence[Sequence[str]],
    min_support: int = 8,
    max_k: int = 2,
    max_terms: int = 120,
) -> dict[tuple[str, ...], set[int]]:
    doc_terms = [set(tokens) for tokens in token_lists]
    doc_freq = Counter()

    for terms in doc_terms:
        doc_freq.update(terms)

    frequent_terms = [term for term, count in doc_freq.most_common(max_terms) if count >= min_support]
    frequent_term_set = set(frequent_terms)
    candidate_counts = Counter()

    for terms in doc_terms:
        filtered_terms = sorted(terms & frequent_term_set)
        for k in range(1, max_k + 1):
            if len(filtered_terms) < k:
                continue
            for termset in combinations(filtered_terms, k):
                candidate_counts[termset] += 1

    candidates = {
        termset: {
            doc_id
            for doc_id, terms in enumerate(doc_terms)
            if set(termset).issubset(terms)
        }
        for termset, support in candidate_counts.items()
        if support >= min_support
    }

    return candidates


def format_termset(termset: tuple[str, ...] | str) -> str:
    if isinstance(termset, str):
        return termset
    return "{" + ", ".join(termset) + "}"


def format_doc_ids(doc_ids: Iterable[int]) -> str:
    return "{" + ",".join(f"D{doc_id + 1}" for doc_id in sorted(doc_ids)) + "}"


def format_doc_frequencies(doc_ids: Iterable[int], doc_candidate_frequency: Counter[int]) -> str:
    return ", ".join(
        f"f{doc_id + 1}={doc_candidate_frequency[doc_id]}"
        for doc_id in sorted(doc_ids)
    )


def make_eo_formula_text(doc_ids: Iterable[int], doc_candidate_frequency: Counter[int]) -> str:
    terms = []
    for doc_id in sorted(doc_ids):
        freq = doc_candidate_frequency[doc_id]
        if freq > 1:
            terms.append(f"((-1/{freq})*LN(1/{freq}))")
    return "EO = " + " + ".join(terms) if terms else "EO = 0"


def entropy_overlap(candidate_docs: set[int], doc_candidate_frequency: Counter[int]) -> float:
    value = 0.0
    for doc_id in candidate_docs:
        freq = doc_candidate_frequency[doc_id]
        if freq > 1:
            value += np.log(freq) / freq
    return float(value)


def ftc_clustering(
    token_lists: Sequence[Sequence[str]],
    min_support: int = 8,
    max_k: int = 2,
    max_terms: int = 120,
) -> tuple[np.ndarray, pd.DataFrame, pd.DataFrame]:
    candidates = generate_frequent_termsets(
        token_lists,
        min_support=min_support,
        max_k=max_k,
        max_terms=max_terms,
    )

    labels = np.full(len(token_lists), -1, dtype=int)
    remaining_docs = set(range(len(token_lists)))
    selected_rows = []
    candidate_rows = []
    cluster_id = 0
    iteration = 1

    while candidates and remaining_docs:
        active_candidates = {
            termset: docs & remaining_docs
            for termset, docs in candidates.items()
        }
        active_candidates = {
            termset: docs
            for termset, docs in active_candidates.items()
            if len(docs) >= min_support
        }

        if not active_candidates:
            break

        doc_candidate_frequency = Counter()
        for docs in active_candidates.values():
            doc_candidate_frequency.update(docs)

        scored_candidates = []
        ordered_candidates = sorted(
            active_candidates.items(),
            key=lambda item: (len(item[0]), item[0]),
        )
        for candidate_number, (termset, docs) in enumerate(ordered_candidates, start=1):
            eo_value = entropy_overlap(docs, doc_candidate_frequency)
            candidate_rows.append(
                {
                    "iteration": iteration,
                    "no_cluster": candidate_number,
                    "frequent_term_set": format_termset(termset),
                    "cluster_candidate": format_doc_ids(docs),
                    "support": len(docs),
                    "doc_frequency": format_doc_frequencies(docs, doc_candidate_frequency),
                    "eo_formula": make_eo_formula_text(docs, doc_candidate_frequency),
                    "eo": round(eo_value, 6),
                }
            )
            scored_candidates.append((eo_value, -len(docs), -len(termset), termset, docs))

        eo_value, neg_support, neg_len, best_termset, best_docs = min(scored_candidates)
        support = len(best_docs)

        labels[list(best_docs)] = cluster_id
        selected_rows.append(
            {
                "iteration": iteration,
                "cluster_id": cluster_id,
                "frequent_termset": ", ".join(best_termset),
                "support": support,
                "entropy_overlap": round(eo_value, 6),
                "n_remaining_docs_after": len(remaining_docs - best_docs),
            }
        )

        remaining_docs -= best_docs
        cluster_id += 1
        iteration += 1

    if remaining_docs:
        residual_cluster_id = cluster_id
        labels[list(remaining_docs)] = residual_cluster_id
        selected_rows.append(
            {
                "iteration": iteration,
                "cluster_id": residual_cluster_id,
                "frequent_termset": "RESIDUAL_UNASSIGNED",
                "support": len(remaining_docs),
                "entropy_overlap": 0.0,
                "n_remaining_docs_after": 0,
            }
        )

    return labels, pd.DataFrame(selected_rows), pd.DataFrame(candidate_rows)


def topic_label(topic_words_df: pd.DataFrame, topic_id: int, n_words: int = 5) -> str:
    words = (
        topic_words_df.loc[topic_words_df["topic_id"] == topic_id]
        .sort_values("rank")
        .head(n_words)["term"]
        .tolist()
    )
    return ", ".join(words)


def make_cluster_summary(
    comments_df: pd.DataFrame,
    topic_words_df: pd.DataFrame,
    token_lists: Sequence[Sequence[str]],
    ftc_selected_clusters: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    termset_lookup = dict(
        zip(ftc_selected_clusters["cluster_id"], ftc_selected_clusters["frequent_termset"])
    )
    eo_lookup = dict(
        zip(ftc_selected_clusters["cluster_id"], ftc_selected_clusters["entropy_overlap"])
    )

    for cluster_id, group in comments_df.groupby("cluster_id"):
        dominant_topic = int(group["topic_id"].mode().iloc[0])
        token_counter = Counter()

        for index in group.index:
            token_counter.update(token_lists[index])

        rows.append(
            {
                "cluster_id": int(cluster_id),
                "n_comments": int(len(group)),
                "ftc_termset": termset_lookup.get(int(cluster_id), ""),
                "entropy_overlap": eo_lookup.get(int(cluster_id), 0.0),
                "dominant_topic": dominant_topic,
                "dominant_topic_terms": topic_label(topic_words_df, dominant_topic),
                "cluster_top_terms": ", ".join(term for term, _ in token_counter.most_common(10)),
                "example_comment": str(group[TEXT_COL].iloc[0]),
            }
        )

    return pd.DataFrame(rows).sort_values("cluster_id").reset_index(drop=True)


In [9]:
cluster_labels, ftc_selected_clusters, ftc_candidate_table = ftc_clustering(
    token_lists,
    min_support=FTC_MIN_SUPPORT,
    max_k=FTC_MAX_K,
    max_terms=FTC_MAX_TERMS,
)

result_df = df.drop(columns=["tokens"]).copy()
result_df["topic_id"] = lda_model.doc_topic_.argmax(axis=1)
result_df["topic_confidence"] = lda_model.doc_topic_.max(axis=1).round(4)
result_df["cluster_id"] = cluster_labels

for topic_id in range(N_TOPICS):
    result_df[f"topic_{topic_id}_prob"] = lda_model.doc_topic_[:, topic_id].round(4)

cluster_summary = make_cluster_summary(
    result_df,
    topic_words,
    token_lists,
    ftc_selected_clusters,
)

ftc_iteration_1_table = ftc_candidate_table.loc[
    ftc_candidate_table["iteration"] == 1,
    ["no_cluster", "frequent_term_set", "cluster_candidate", "eo"],
].rename(
    columns={
        "no_cluster": "No Cluster",
        "frequent_term_set": "Frequent term set",
        "cluster_candidate": "Cluster candidate",
        "eo": "EO",
    }
)

print("Jumlah cluster FTC:", result_df["cluster_id"].nunique())
print("Min support FTC:", FTC_MIN_SUPPORT)
print("Max k termset:", FTC_MAX_K)
print("Jumlah kandidat FTC iterasi 1:", len(ftc_iteration_1_table))
ftc_iteration_1_table


Jumlah cluster FTC: 46
Min support FTC: 8
Max k termset: 2
Jumlah kandidat FTC iterasi 1: 141


,No Cluster,Frequent term set,Cluster candidate,EO
0,1,{adil},"{D13,D16,D36,D38,D39,D40,D61,D68,D77,D107,D108,D140,D148,D162,D177,D184,D191,D200,D208,D213,D217,D240,D243,D252,D258,D269,D297,D303,D331,D357,D366,D389,D396...",10.209697
1,2,{allah},"{D11,D34,D38,D39,D41,D43,D52,D68,D100,D111,D162,D181,D189,D190,D207,D228,D239,D303,D322,D352,D357,D394,D396,D399,D408,D442,D455,D486}",6.900040
2,3,{alloh},"{D16,D22,D86,D118,D171,D187,D279,D299,D304,D326,D357,D369}",3.680216
3,4,{anak},"{D29,D150,D162,D187,D194,D241,D258,D261,D270,D365,D388,D401,D417,D434}",4.021828
4,5,{aneh},"{D5,D85,D88,D148,D220,D295,D406,D411}",1.976066
...,...,...,...,...
136,137,"{nadiem, pak}","{D3,D15,D51,D52,D171,D176,D180,D182,D189,D190,D268,D296,D298,D322,D347,D354,D401,D451,D454,D455,D457,D462,D466,D500}",6.148647
137,138,"{nadim, pak}","{D21,D43,D112,D125,D172,D201,D207,D238,D248,D260,D468}",3.063214
138,139,"{orang, pak}","{D3,D15,D17,D40,D207,D248,D322,D401,D429}",1.625457
139,140,"{pak, sabar}","{D3,D36,D40,D43,D135,D180,D351,D363,D414,D455,D466}",2.891575


In [10]:
print("Cluster terpilih FTC per iterasi:")
display(ftc_selected_clusters.head(20))

print("Ringkasan cluster akhir:")
cluster_summary


Cluster terpilih FTC per iterasi:


,iteration,cluster_id,frequent_termset,support,entropy_overlap,n_remaining_docs_after
0,1,0,"hukum, jadi",8,1.590554,492
1,2,1,lalu,8,1.548112,484
2,3,2,"orang, pak",8,1.572495,476
3,4,3,didik,8,1.859656,468
4,5,4,"jadi, nadiem",8,1.927286,460
5,6,5,ngeri,8,2.030486,452
6,7,6,jujur,8,2.125935,444
7,8,7,"jabat, jadi",8,1.992404,436
8,9,8,konoha,8,2.010855,428
9,10,9,lah,9,2.106305,419


Ringkasan cluster akhir:


,cluster_id,n_comments,ftc_termset,entropy_overlap,dominant_topic,dominant_topic_terms,cluster_top_terms,example_comment
0,0,8,"hukum, jadi",1.590554,4,"jadi, tumbal, mau, nadiem, jokowi","jadi, hukum, nadiem, orang, jokowi, hakim, tumbal, tuhan, uda, negara","Hukum sudah menjadi alat kekuasaan,\nIni Uda menjadi negara tiran.\nSudah wajib hukum nya dilakukan revolusi akhlak\nAgar GK menjadi makin banyak makan korb..."
1,1,8,lalu,1.548112,1,"orang, sama, baik, nadim, jangan","lalu, aku, peran, engkau, tubuh, jasmani, orang, panggung, diri, akal",Lalu bagaimana kabar Dadang sekarang
2,2,8,"orang, pak",1.572495,0,"pak, allah, nadiem, adil, moga","pak, orang, bapak, tak, tuhan, jadi, siapa, baik, nadiem, mungkin","Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya"
3,3,8,didik,1.859656,3,"hukum, indonesia, negara, gila, rakyat","didik, jadi, indonesia, kamu, nadiem, moga, mas, besar, lebih, mundur","Begitu besar jasa Nadiem untuk Pendidikan Indonesia...itu jelas , tidak bisa dipungkiri... semoga hukum bicara lebih baik lebih adil.\nNadiem... semoga Alla..."
4,4,8,"jadi, nadiem",1.927286,4,"jadi, tumbal, mau, nadiem, jokowi","nadiem, jadi, jokowi, menteri, tumbal, pak, salah, tahu, astagfirulloh, uang",Kasihan Nadiem..dia korban dari Jokowi..kesalahan Nadiem kenapa dia mau menjadi menteri..
5,5,8,ngeri,2.030486,3,"hukum, indonesia, negara, gila, rakyat","ngeri, indonesia, birokrasi, nadiem, gila, hukum, bro, nepalisasi, konoha, cepat",Ngeri dengan hukum di indonesia
6,6,8,jujur,2.125935,4,"jadi, tumbal, mau, nadiem, jokowi","jujur, orang, jadi, pak, nadiem, masuk, baik, biangkeroknya, tangkap, siapa",Biangkeroknya tangkap! Tapi siapa yang bisa nangkap???😢 semoga Tuhan memberikan hidajah untuk orang2 jujur. Aamiin
7,7,8,"jabat, jadi",1.992404,4,"jadi, tumbal, mau, nadiem, jokowi","jadi, jabat, mau, menteri, negeri, tumbal, anak, jokowi, kaya, orang",Ngapain jadi pejabat mending tolak klw ujung nya menyakitkan
8,8,8,konoha,2.010855,3,"hukum, indonesia, negara, gila, rakyat","konoha, politik, mau, negara, beliau, memang, paling, aneh, dunia, nadiem",Negara Konoha memang negara paling aneh sedunia..
9,9,9,lah,2.106305,1,"orang, sama, baik, nadim, jangan","lah, nadim, gojek, sama, orang, amin, allah, buka, ngapain, jadi",Ya bongkar lah semuaa\nMasa sudah tersudut masih belum dibuka😅


### Visualisasi Hasil LDA

Visualisasi ini menampilkan jumlah komentar pada setiap topik LDA dan kata dominan pada masing-masing topik. Grafik dibuat dengan HTML/CSS agar notebook tetap berjalan tanpa dependensi tambahan seperti `matplotlib`.


In [11]:
class HTMLBlock:
    def __init__(self, html: str):
        self.html = html

    def _repr_html_(self) -> str:
        return self.html

    def __repr__(self) -> str:
        return "HTML visualization generated."


def html_escape(value: object) -> str:
    return (
        str(value)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
    )


def make_topic_count_bar_html(topic_count_df: pd.DataFrame) -> str:
    max_count = max(1, int(topic_count_df["n_comments"].max()))
    rows = []
    for _, row in topic_count_df.iterrows():
        width = (float(row["n_comments"]) / max_count) * 100
        rows.append(
            f"""
            <div class='bar-row'>
                <div class='bar-label'>Topic {int(row['topic_id'])}</div>
                <div class='bar-track'>
                    <div class='bar-fill topic-fill' style='width:{width:.2f}%'></div>
                </div>
                <div class='bar-value'>{int(row['n_comments'])} komentar</div>
            </div>
            <div class='topic-terms'>{html_escape(row['topic_terms'])}</div>
            """
        )
    return "".join(rows)


def make_topic_words_bar_html(topic_words_df: pd.DataFrame, top_n: int = 8) -> str:
    topic_sections = []
    for topic_id in sorted(topic_words_df["topic_id"].unique()):
        words = (
            topic_words_df.loc[topic_words_df["topic_id"] == topic_id]
            .sort_values("rank")
            .head(top_n)
            .copy()
        )
        max_probability = max(1e-12, float(words["probability"].max()))
        word_rows = []
        for _, row in words.iterrows():
            width = (float(row["probability"]) / max_probability) * 100
            word_rows.append(
                f"""
                <div class='word-row'>
                    <div class='word-label'>{html_escape(row['term'])}</div>
                    <div class='bar-track small'>
                        <div class='bar-fill word-fill' style='width:{width:.2f}%'></div>
                    </div>
                    <div class='word-value'>{float(row['probability']):.4f}</div>
                </div>
                """
            )
        topic_sections.append(
            f"""
            <section class='topic-card'>
                <h4>Topic {int(topic_id)}</h4>
                {''.join(word_rows)}
            </section>
            """
        )
    return "<div class='topic-grid'>" + "".join(topic_sections) + "</div>"


topic_count_table = (
    result_df["topic_id"]
    .value_counts()
    .sort_index()
    .rename_axis("topic_id")
    .reset_index(name="n_comments")
)
topic_count_table["topic_terms"] = topic_count_table["topic_id"].apply(
    lambda topic_id: topic_label(topic_words, int(topic_id), n_words=6)
)

lda_visualization_html = f"""
    <style>
        .lda-viz {{font-family: Arial, sans-serif; color: #172033; max-width: 980px;}}
        .lda-viz h3 {{margin: 18px 0 10px; color: #0f4c81;}}
        .bar-row, .word-row {{display: grid; grid-template-columns: 92px 1fr 110px; gap: 10px; align-items: center; margin: 7px 0;}}
        .bar-label, .word-label {{font-weight: 600; font-size: 13px;}}
        .bar-track {{height: 22px; background: #eef3f8; border: 1px solid #d7e0ea; border-radius: 4px; overflow: hidden;}}
        .bar-track.small {{height: 18px;}}
        .bar-fill {{height: 100%;}}
        .topic-fill {{background: #1f77b4;}}
        .word-fill {{background: #2ca02c;}}
        .bar-value, .word-value {{font-size: 12px; color: #3d4a5c; text-align: right;}}
        .topic-terms {{margin: -3px 0 10px 102px; color: #59677a; font-size: 12px;}}
        .topic-grid {{display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 14px;}}
        .topic-card {{border: 1px solid #d7e0ea; border-radius: 6px; padding: 12px; background: #ffffff;}}
        .topic-card h4 {{margin: 0 0 8px; color: #0f4c81;}}
    </style>
    <div class='lda-viz'>
        <h3>Bar Chart Jumlah Komentar per Topik LDA</h3>
        {make_topic_count_bar_html(topic_count_table)}
        <h3>Bar Chart Top Words per Topic</h3>
        {make_topic_words_bar_html(topic_words, top_n=8)}
    </div>
"""

topic_count_table.to_csv(OUTPUT_DIR / "lda_topic_comment_counts.csv", index=False)
(OUTPUT_DIR / "lda_visualisasi_bar_chart.html").write_text(lda_visualization_html, encoding="utf-8")

print("Visualisasi LDA dibuat:")
print("- outputs/lda_topic_comment_counts.csv")
print("- outputs/lda_visualisasi_bar_chart.html")
display(HTMLBlock(lda_visualization_html))


Visualisasi LDA dibuat:
- outputs/lda_topic_comment_counts.csv
- outputs/lda_visualisasi_bar_chart.html


HTML visualization generated.

In [12]:
result_df[[TEXT_COL, "clean_text", "topic_id", "topic_confidence", "cluster_id"]].head(20)


,komentar,clean_text,topic_id,topic_confidence,cluster_id
0,"Org sepintar ini,kalah saya tukang mebel.😢😢😢",orang pintar kalah tukang mebel,1,0.8857,37
1,Perlawanan ojol dgn tolak jokodok,lawan ojol tolak jokodok,2,0.4667,45
2,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",pilpres nasib juta orang indonesia atur cuma orang indonesia pak nadiem sabar lah,1,0.4080,2
3,Kejahatan Luarbiasa DiNegeri ini,jahat luarbiasa negeri,3,0.8400,12
4,Negara Konoha memang negara paling aneh sedunia..,negara konoha memang negara paling aneh dunia,3,0.7846,8
5,"Yg membuka lapangan kerja ber juta orang di Indonesia di bui,sedangkan koruptor di Indonesia yg merajalela,semakin solid,(pasti bung Karno&Soeharto)sedih me...",buka lapang kerja ber juta orang indonesia bui koruptor indonesia rajalela makin solid bung karno soeharto sedih lihat rakyat indonesia sengsara,3,0.9758,26
6,Biangkeroknya tangkap! Tapi siapa yang bisa nangkap???😢 semoga Tuhan memberikan hidajah untuk orang2 jujur. Aamiin,biangkeroknya tangkap siapa nangkap moga tuhan beri hidajah orang jujur aamiin,0,0.5368,6
7,Yg Korupsi MBG tuh...di Hukum Mati,korupsi mbg tuh hukum mati,2,0.5636,31
8,Kenapa begitu.. coba dehh tolong menolong logikanya egois .. yang menolong jangan setengah-setengah..,coba dehh logika egois jangan tengah tengah,1,0.4400,20
9,Astaghfirullah,astaghfirullah,0,0.7333,45


## 4. Export Hasil

Cell ini menyimpan hasil agar mudah dipakai untuk laporan atau analisis lanjutan.

In [13]:
result_df.to_csv(OUTPUT_DIR / "comment_topics_clusters.csv", index=False)

preprocessed_columns = ["comment_id", TEXT_COL, "normalized_text", "clean_text", "token_count"]
result_df[preprocessed_columns].to_csv(
    OUTPUT_DIR / "comments_preprocessed.csv",
    index=False,
)

topic_words.to_csv(OUTPUT_DIR / "topic_words.csv", index=False)
topic_matrix.to_csv(OUTPUT_DIR / "topic_matrix.csv", index=False)
cluster_summary.to_csv(OUTPUT_DIR / "cluster_summary.csv", index=False)
ftc_selected_clusters.to_csv(OUTPUT_DIR / "ftc_selected_clusters.csv", index=False)
ftc_candidate_table.to_csv(OUTPUT_DIR / "ftc_candidate_table.csv", index=False)

print("Output tersimpan di:", OUTPUT_DIR.resolve())
print("- comments_preprocessed.csv")
print("- topic_words.csv")
print("- topic_matrix.csv")
print("- comment_topics_clusters.csv")
print("- cluster_summary.csv")
print("- ftc_selected_clusters.csv")
print("- ftc_candidate_table.csv")


Output tersimpan di: C:\Users\fariz\OneDrive\Desktop\UAS-TOPIK\outputs
- comments_preprocessed.csv
- topic_words.csv
- topic_matrix.csv
- comment_topics_clusters.csv
- cluster_summary.csv
- ftc_selected_clusters.csv
- ftc_candidate_table.csv


## 5. Rule-Based NER

NER memakai pendekatan rule-based. Tahapannya dibuat eksplisit seperti materi NER: daftar fitur kontekstual, fitur morfologis, fitur part-of-speech sederhana, lalu tokenisasi dan assignment fitur sebelum entity dikenali dengan kamus/pola regex.


In [14]:
# Daftar pola (pattern) untuk Rule-Based Named Entity Recognition (NER)
RULE_BASED_NER_PATTERNS = [
    ("PER", "person_nadiem", r"\b(nadiem makarim|nadiem|nadim|nadzim)\b"),
    ("PER", "person_jokowi", r"\b(jokowi|jokodok|mulyono)\b"),
    ("PER", "person_public", r"\b(prabowo|gibran|ahok|hotman)\b"),
    ("ORG", "org_gojek", r"\b(gojek|goto|kemendikbud|ojol)\b"),
    ("GPE", "geo_indonesia", r"\b(indonesia|ri|jawa|jakarta)\b"),
    ("LAW", "legal_terms", r"\b(hukum|hakim|jaksa|putusan|vonis|penjara|denda|bukti|pengadilan|korupsi|koruptor)\b"),
    ("REG", "religion_terms", r"\b(allah|alloh|tuhan|insyaallah|alhamdulillah|adzab)\b"),
    ("EVT", "event_terms", r"\b(pilpres|demo|pemilu)\b"),
    ("MON", "money_terms", r"\b(rp\s?\d+[\d.,]*|\d+\s?(miliar|milyar|juta|ribu))\b"),
    ("QTY", "quantity_terms", r"\b\d+\s?(tahun|orang|persen|%)\b"),
]

# Fungsi untuk menjalankan Rule-Based NER
def run_rule_based_ner(texts: Iterable[object], comment_ids: Iterable[object] | None = None) -> pd.DataFrame:

     # Menyimpan seluruh hasil entity yang ditemukan
    rows = []
    comment_id_list = list(comment_ids) if comment_ids is not None else list(range(len(list(texts))))
    text_values = ["" if pd.isna(text) else str(text) for text in texts]

    for doc_id, text in enumerate(text_values):

        seen_spans = set()
        for label, rule_name, pattern in RULE_BASED_NER_PATTERNS:

             # Mencari semua kecocokan regex pada teks (tidak sensitif huruf besar-kecil)
            for match in re.finditer(pattern, text, flags=re.IGNORECASE):
                span_key = (match.start(), match.end(), label)
                if span_key in seen_spans:
                    continue
                seen_spans.add(span_key)

                 # Mencari semua kecocokan regex pada teks (tidak sensitif huruf besar-kecil)
                rows.append(
                    {
                        "doc_id": doc_id,
                        "comment_id": comment_id_list[doc_id],
                        "text": text,
                        "entity": match.group(0),
                        "label": label,
                        "rule_name": rule_name,
                        "score": 1.0,
                        "start": match.start(),
                        "end": match.end(),
                    }
                )

#ubah listnya ke df pandas
    return pd.DataFrame(rows)


### 5.1 Tahapan Feature Extraction Rule-Based NER

Cell ini menampilkan tabel fitur seperti contoh materi: contextual features, morphological features, part-of-speech features, dan hasil tokenisasi beserta assignment fitur untuk satu komentar contoh dari dataset.


In [15]:
# Membuat tabel daftar fitur kontekstual beserta penjelasan dan contohnya
contextual_feature_table = pd.DataFrame(

     # Data fitur kontekstual
    [
        ("PPRE", "Person prefix", "Dr., Pak, Bpk, K.H."),
        ("PMID", "Person middle", "bin, van"),
        ("PSUF", "Person suffix", "S.Kom, SH"),
        ("PTIT", "Person title", "Menteri, Presiden, Pak"),
        ("OPRE", "Organization prefix", "PT, Universitas, Kementerian"),
        ("OSUF", "Organization suffix", "Ltd., Company, Tbk"),
        ("OPOS", "Position in organization", "Ketua, Menteri, Rektor"),
        ("OCON", "Other organization contextual", "Mukernas, Rakernas, Gojek"),
        ("LPRE", "Location prefix", "Kota, Provinsi, Kabupaten"),
        ("LSUF", "Location suffix", "Utara, Selatan, City"),
        ("LLDR", "Location leader", "Gubernur, Walikota"),
        ("POLP", "Prepositions usually followed by person name", "oleh, untuk"),
        ("LOPP", "Prepositions usually followed by location name", "di, ke, dari"),
        ("DAY", "Day", "Senin, Sabtu"),
        ("MONTH", "Month", "April, Mei"),
    ],

    #simpan ke 
    columns=["Feature Name", "Explanation", "Example"],
)

morphological_feature_table = pd.DataFrame(
    [
        ("TitleCase", "Begin with uppercase letter and followed by lowercase letter", "Soedirman"),
        ("UpperCase", "All uppercase letter", "KPU"),
        ("LowerCase", "All lowercase letter", "menuntut"),
        ("MixedCase", "Uppercase and lowercase letter", "LeIP"),
        ("CapStart", "Begin with uppercase letter", "LeIP, Muhammad"),
        ("CharDigit", "Letter and number", "P3K"),
        ("Digit", "All number", "2004"),
        ("DigitSlash", "Number with slash", "17/5"),
        ("Numeric", "Number with dot or comma", "20,5; 17.500,00"),
        ("NumStr", "Number in word", "satu, tujuh, lima"),
        ("Roman", "Roman number", "VII, XI"),
        ("TimeForm", "Number in time format", "17:05, 19.30"),
    ],

     # Menentukan nama kolom pada DataFrame
    columns=["Feature Name", "Explanation", "Example"],
)


# Membuat tabel daftar fitur morfologi beserta penjelasan dan contohnya
pos_feature_table = pd.DataFrame(
    [

         # Data fitur part-of-speech
        ("ART", "Article", "si, sang"),
        ("ADJ", "Adjective", "indah, baik, benar"),
        ("ADV", "Adverb", "telah, kemarin, sudah"),
        ("AUX", "Auxiliary verb", "harus, bisa"),
        ("C", "Conjunction", "dan, atau, lalu"),
        ("DEF", "Definition", "merupakan, adalah"),
        ("NOUN", "Noun", "rumah, hukum, korupsi"),
        ("NOUNP", "Personal noun", "ayah, ibu, pak"),
        ("NUM", "Number", "satu, dua, 500"),
        ("MODAL", "Modal", "akan, mau"),
        ("OOV", "Out of dictionary", "jokodok"),
        ("PAR", "Particle", "kah, pun"),
        ("PREP", "Preposition", "di, ke, dari"),
        ("PRO", "Pronominal", "saya, beliau"),
        ("VACT", "Active verb", "menuduh, menuntut"),
        ("VPAS", "Passive verb", "dituduh, dihukum"),
        ("VERB", "Verb", "pergi, dukung"),
    ],
    columns=["Feature Name", "Explanation", "Example"],
)

# Menentukan nama kolom pada DataFrame

PERSON_PREFIX = {"dr", "pak", "bpk", "k.h", "kh", "prof", "ir"}
PERSON_MIDDLE = {"bin", "binti", "van"}
PERSON_SUFFIX = {"sh", "s.h", "skom", "s.kom", "mh", "m.h"}
PERSON_TITLE = {"pak", "bapak", "ibu", "menteri", "presiden", "hakim", "jaksa"}
ORG_PREFIX = {"pt", "universitas", "kementerian", "kemendikbud", "partai"}
ORG_SUFFIX = {"tbk", "ltd", "company"}
ORG_POSITION = {"ketua", "menteri", "rektor", "direktur"}
ORG_CONTEXT = {"mukernas", "rakernas", "gojek", "goto", "ojol", "kpu"}
LOCATION_PREFIX = {"kota", "provinsi", "kabupaten"}
LOCATION_SUFFIX = {"utara", "selatan", "barat", "timur", "city"}
LOCATION_LEADER = {"gubernur", "walikota", "bupati"}
PERSON_PREPOSITIONS = {"oleh", "untuk"}
LOCATION_PREPOSITIONS = {"di", "ke", "dari"}
DAY_WORDS = {"senin", "selasa", "rabu", "kamis", "jumat", "sabtu", "minggu"}
MONTH_WORDS = {"januari", "februari", "maret", "april", "mei", "juni", "juli", "agustus", "september", "oktober", "november", "desember"}
NUMBER_WORDS = {"nol", "satu", "dua", "tiga", "empat", "lima", "enam", "tujuh", "delapan", "sembilan", "sepuluh", "ratus", "ribu", "juta", "miliar", "milyar"}
ROMAN_WORDS = {"I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X", "XI", "XII"}


# Kamus sederhana untuk fitur part-of-speech
POS_LEXICON = {
    "ART": {"si", "sang"},
    "ADJ": {"baik", "benar", "bener", "buruk", "gila", "adil", "salah"},
    "ADV": {"telah", "kemarin", "sudah", "belum", "sekarang", "nanti", "semakin"},
    "AUX": {"harus", "bisa", "dapat", "mesti"},
    "C": {"dan", "atau", "lalu", "tetapi", "karena", "sedangkan"},
    "DEF": {"adalah", "merupakan", "yaitu"},
    "NOUN": {"rumah", "hukum", "korupsi", "koruptor", "negara", "rakyat", "uang", "omset", "menteri", "hakim", "jaksa", "penjara"},
    "NOUNP": {"ayah", "ibu", "pak", "bapak", "beliau"},
    "MODAL": {"akan", "mau", "ingin"},
    "PAR": {"kah", "pun", "lah"},
    "PREP": {"di", "ke", "dari", "pada", "untuk", "oleh"},
    "PRO": {"saya", "aku", "kamu", "dia", "beliau", "mereka", "kita", "kami"},
    "VERB": {"pergi", "dukung", "tolak", "tangkap", "masuk", "bui", "pilih", "atur"},
}

# Pola regex untuk tokenisasi teks
TOKEN_PATTERN = re.compile(r"\d+(?:[/:.,]\d+)*|[A-Za-z]+(?:[-'][A-Za-z]+)*|\w+|[^\w\s]", re.UNICODE)

# Fungsi untuk menggabungkan daftar fitur menjadi string
def join_features(features: list[str]) -> str:
    return ", ".join(features)

# Fungsi untuk melakukan tokenisasi pada teks
def tokenize_for_ner_features(text: object) -> list[str]:
    if pd.isna(text):
        return []
    return TOKEN_PATTERN.findall(str(text))

# Fungsi untuk menentukan jenis token
def token_kind(token: str) -> str:
    if re.fullmatch(r"\d+(?:[/:.,]\d+)*", token):
        return "NUM"
    if re.fullmatch(r"\w+", token, flags=re.UNICODE):
        return "WORD"
    if token == "(":
        return "SPUNC"
    if token == ")":
        return "EPUNC"
    return "OPUNC"

# Fungsi untuk menentukan fitur kontekstual pada token tertentu
def assign_contextual_features(tokens: list[str], index: int) -> str:
    token = tokens[index]
    lower = token.lower().strip(".")
    features = []

     # Mengelompokkan nama fitur dengan kumpulan kata yang sesuai
    feature_sets = [
        ("PPRE", PERSON_PREFIX), ("PMID", PERSON_MIDDLE), ("PSUF", PERSON_SUFFIX), ("PTIT", PERSON_TITLE),
        ("OPRE", ORG_PREFIX), ("OSUF", ORG_SUFFIX), ("OPOS", ORG_POSITION), ("OCON", ORG_CONTEXT),
        ("LPRE", LOCATION_PREFIX), ("LSUF", LOCATION_SUFFIX), ("LLDR", LOCATION_LEADER),
        ("POLP", PERSON_PREPOSITIONS), ("LOPP", LOCATION_PREPOSITIONS), ("DAY", DAY_WORDS), ("MONTH", MONTH_WORDS),
    ]
    for feature_name, values in feature_sets:
        if lower in values:
            features.append(feature_name)
    return join_features(features)


def assign_morphological_features(token: str) -> str:
    features = []
    if re.fullmatch(r"\d+", token):
        features.append("Digit")
    if re.fullmatch(r"\d+/\d+", token):
        features.append("DigitSlash")
    if re.fullmatch(r"\d+[.,]\d+(?:[.,]\d+)*", token):
        features.append("Numeric")
    if re.fullmatch(r"\d{1,2}[:.]\d{2}", token):
        features.append("TimeForm")
    if token.lower() in NUMBER_WORDS:
        features.append("NumStr")
    if token.upper() in ROMAN_WORDS:
        features.append("Roman")
    if any(ch.isalpha() for ch in token) and any(ch.isdigit() for ch in token):
        features.append("CharDigit")
    if token.isalpha():
        if token.istitle():
            features.append("TitleCase")
        if token.isupper() and len(token) > 1:
            features.append("UpperCase")
        if token.islower():
            features.append("LowerCase")
        if not token.islower() and not token.isupper() and not token.istitle():
            features.append("MixedCase")
        if token[0].isupper():
            features.append("CapStart")
    return join_features(features)


def assign_pos_features(token: str) -> str:
    lower = token.lower()
    features = []
    if re.fullmatch(r"\d+(?:[/:.,]\d+)*", token) or lower in NUMBER_WORDS:
        features.append("NUM")
    for tag, words in POS_LEXICON.items():
        if lower in words:
            features.append(tag)
    if lower.startswith("men") or lower.startswith("mem") or lower.startswith("meng") or lower.startswith("me"):
        features.append("VACT")
    if lower.startswith("di") and len(lower) > 3 and lower not in {"di", "dia"}:
        features.append("VPAS")
    if token_kind(token) != "WORD" and token_kind(token) != "NUM":
        return ""
    if not features:
        features.append("OOV")
    return join_features(sorted(set(features), key=features.index))


def build_token_feature_assignment_table(text: object) -> pd.DataFrame:
    tokens = tokenize_for_ner_features(text)
    return pd.DataFrame(
        [
            {
                "Token string": token,
                "Token kind": token_kind(token),
                "Contextual features": assign_contextual_features(tokens, index),
                "Morphological features": assign_morphological_features(token),
                "Part-of-speech features": assign_pos_features(token),
            }
            for index, token in enumerate(tokens)
        ]
    )


def build_contextual_feature_if_then_table() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "Rule ID": "CF1_PERSON_PREFIX",
                "IF": "Token[i].Kind='WORD' AND lower(Token[i]) IN {dr, pak, bpk, kh, prof, ir}",
                "THEN": "Token[i].ContextualFeature='PPRE'",
                "Contoh": "Pak -> PPRE",
            },
            {
                "Rule ID": "CF2_PERSON_TITLE",
                "IF": "Token[i].Kind='WORD' AND lower(Token[i]) IN {menteri, presiden, hakim, jaksa}",
                "THEN": "Token[i].ContextualFeature='PTIT'",
                "Contoh": "Menteri -> PTIT",
            },
            {
                "Rule ID": "CF3_ORG_POSITION",
                "IF": "Token[i].Kind='WORD' AND lower(Token[i]) IN {ketua, menteri, rektor, direktur}",
                "THEN": "Token[i].ContextualFeature='OPOS'",
                "Contoh": "Ketua -> OPOS",
            },
            {
                "Rule ID": "CF4_ORG_CONTEXT",
                "IF": "Token[i].Kind='WORD' AND lower(Token[i]) IN {gojek, goto, ojol, kpu, kemendikbud}",
                "THEN": "Token[i].ContextualFeature='OCON'",
                "Contoh": "Gojek -> OCON",
            },
            {
                "Rule ID": "CF5_LOCATION_PREPOSITION",
                "IF": "Token[i].Kind='WORD' AND lower(Token[i]) IN {di, ke, dari}",
                "THEN": "Token[i].ContextualFeature='LOPP'",
                "Contoh": "di -> LOPP",
            },
            {
                "Rule ID": "CF6_DATE_CONTEXT",
                "IF": "Token[i].Kind='WORD' AND lower(Token[i]) IN daftar hari/bulan",
                "THEN": "Token[i].ContextualFeature='DAY' atau 'MONTH'",
                "Contoh": "Senin -> DAY; Mei -> MONTH",
            },
        ]
    )


def build_morphological_feature_if_then_table() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "Rule ID": "MF1_TITLE_CASE",
                "IF": "Token[i] begins with uppercase letter AND remaining letters are lowercase",
                "THEN": "Token[i].MorphologicalFeature='TitleCase' and 'CapStart'",
                "Contoh": "Nadiem -> TitleCase, CapStart",
            },
            {
                "Rule ID": "MF2_UPPER_CASE",
                "IF": "Token[i] consists of uppercase letters AND length(Token[i]) > 1",
                "THEN": "Token[i].MorphologicalFeature='UpperCase' and 'CapStart'",
                "Contoh": "KPU -> UpperCase, CapStart",
            },
            {
                "Rule ID": "MF3_LOWER_CASE",
                "IF": "Token[i] consists of lowercase letters",
                "THEN": "Token[i].MorphologicalFeature='LowerCase'",
                "Contoh": "hukum -> LowerCase",
            },
            {
                "Rule ID": "MF4_DIGIT",
                "IF": "Token[i] consists only of numbers",
                "THEN": "Token[i].MorphologicalFeature='Digit'",
                "Contoh": "2004 -> Digit",
            },
            {
                "Rule ID": "MF5_DIGIT_SLASH",
                "IF": "Token[i] matches number/number pattern",
                "THEN": "Token[i].MorphologicalFeature='DigitSlash'",
                "Contoh": "17/5 -> DigitSlash",
            },
            {
                "Rule ID": "MF6_NUMERIC",
                "IF": "Token[i] is a number containing dot or comma",
                "THEN": "Token[i].MorphologicalFeature='Numeric'",
                "Contoh": "16,3 -> Numeric",
            },
        ]
    )


def build_pos_feature_if_then_table() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "Rule ID": "PF1_PREPOSITION",
                "IF": "Token[i].Kind='WORD' AND lower(Token[i]) IN {di, ke, dari, pada, untuk, oleh}",
                "THEN": "Token[i].POSFeature='PREP'",
                "Contoh": "dari -> PREP",
            },
            {
                "Rule ID": "PF2_CONJUNCTION",
                "IF": "Token[i].Kind='WORD' AND lower(Token[i]) IN {dan, atau, lalu, tetapi, karena}",
                "THEN": "Token[i].POSFeature='C'",
                "Contoh": "dan -> C",
            },
            {
                "Rule ID": "PF3_NOUN",
                "IF": "Token[i].Kind='WORD' AND lower(Token[i]) IN kamus NOUN",
                "THEN": "Token[i].POSFeature='NOUN'",
                "Contoh": "hukum -> NOUN",
            },
            {
                "Rule ID": "PF4_NUMBER",
                "IF": "Token[i].Kind='NUM' OR lower(Token[i]) IN daftar kata bilangan",
                "THEN": "Token[i].POSFeature='NUM'",
                "Contoh": "280 -> NUM; juta -> NUM",
            },
            {
                "Rule ID": "PF5_ACTIVE_VERB",
                "IF": "lower(Token[i]) starts with {men, mem, meng, me}",
                "THEN": "Token[i].POSFeature='VACT'",
                "Contoh": "menuduh -> VACT",
            },
            {
                "Rule ID": "PF6_OUT_OF_DICTIONARY",
                "IF": "Token[i].Kind='WORD' AND Token[i] does not match any POS lexicon/rule",
                "THEN": "Token[i].POSFeature='OOV'",
                "Contoh": "jokodok -> OOV",
            },
        ]
    )


def build_if_then_ner_rules_table() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "Rule ID": "R1_ORG_AFTER_POSITION",
                "IF": "Token[i].Kind='WORD' AND Token[i].OPOS AND Token[i+1].Kind='WORD' AND (Token[i+1].TitleCase OR Token[i+1].CapStart OR Token[i+1].OCON)",
                "THEN": "Token[i+1].NE='ORGANIZATION'",
                "Contoh": "Ketua Gojek / Menteri Kemendikbud",
            },
            {
                "Rule ID": "R2_PERSON_AFTER_PREFIX",
                "IF": "Token[i].Kind='WORD' AND (Token[i].PPRE OR Token[i].PTIT) AND Token[i+1].Kind='WORD' AND (Token[i+1].TitleCase OR Token[i+1].CapStart)",
                "THEN": "Token[i+1].NE='PERSON'",
                "Contoh": "Pak Nadiem / Bpk Jokowi",
            },
            {
                "Rule ID": "R3_LOCATION_AFTER_PREPOSITION",
                "IF": "Token[i].Kind='WORD' AND Token[i].LOPP AND Token[i+1].Kind='WORD' AND (Token[i+1].TitleCase OR Token[i+1].CapStart)",
                "THEN": "Token[i+1].NE='LOCATION'",
                "Contoh": "di Indonesia / ke Jakarta",
            },
            {
                "Rule ID": "R4_MONEY_QUANTITY",
                "IF": "Token[i].Kind='NUM' AND Token[i+1].Kind='WORD' AND Token[i+1] IN {ribu, juta, miliar, milyar, triliun}",
                "THEN": "Token[i:i+1].NE='MONEY'",
                "Contoh": "280 juta / 16,3 triliun",
            },
            {
                "Rule ID": "R5_LEGAL_TERM",
                "IF": "Token[i].Kind='WORD' AND Token[i] IN {hukum, hakim, jaksa, putusan, vonis, penjara, denda, korupsi, koruptor}",
                "THEN": "Token[i].NE='LAW'",
                "Contoh": "hakim / vonis / korupsi",
            },
            {
                "Rule ID": "R6_RELIGION_TERM",
                "IF": "Token[i].Kind='WORD' AND Token[i] IN {allah, alloh, tuhan, insyaallah, alhamdulillah, adzab}",
                "THEN": "Token[i].NE='RELIGION'",
                "Contoh": "Allah / Tuhan",
            },
        ]
    )


def render_if_then_rules_html(rule_table: pd.DataFrame) -> str:
    blocks = []
    for _, row in rule_table.iterrows():
        blocks.append(
            f"""
            <div style='border:1px solid #c7d2e3; border-radius:6px; margin:10px 0; padding:10px 12px; font-family:Arial, sans-serif;'>
                <div style='font-weight:700; color:#173B7A; margin-bottom:6px;'>{html_escape(row['Rule ID'])}</div>
                <div style='display:grid; grid-template-columns:64px 1fr; gap:8px; margin:4px 0;'>
                    <strong>IF</strong><code style='white-space:normal;'>{html_escape(row['IF'])}</code>
                    <strong>THEN</strong><code style='white-space:normal;'>{html_escape(row['THEN'])}</code>
                    <strong>Contoh</strong><span>{html_escape(row['Contoh'])}</span>
                </div>
            </div>
            """
        )
    return "".join(blocks)


def select_ner_step_sample(texts: Iterable[object]) -> str:
    best_text = ""
    best_score = -1
    for text in texts:
        text_value = "" if pd.isna(text) else str(text)
        tokens = tokenize_for_ner_features(text_value)
        pattern_hits = sum(len(re.findall(pattern, text_value, flags=re.IGNORECASE)) for _, _, pattern in RULE_BASED_NER_PATTERNS)
        contextual_hits = sum(bool(assign_contextual_features(tokens, i)) for i in range(len(tokens)))
        score = (pattern_hits * 5) + (contextual_hits * 2) + min(len(tokens), 30)
        if score > best_score:
            best_score = score
            best_text = text_value
    return best_text


sample_text_for_ner_steps = select_ner_step_sample(result_df[TEXT_COL].tolist())
ner_token_feature_assignment = build_token_feature_assignment_table(sample_text_for_ner_steps)
contextual_if_then_rules_table = build_contextual_feature_if_then_table()
morphological_if_then_rules_table = build_morphological_feature_if_then_table()
pos_if_then_rules_table = build_pos_feature_if_then_table()
ner_if_then_rules_table = build_if_then_ner_rules_table()

contextual_feature_table.to_csv(OUTPUT_DIR / "ner_contextual_features.csv", index=False)
morphological_feature_table.to_csv(OUTPUT_DIR / "ner_morphological_features.csv", index=False)
pos_feature_table.to_csv(OUTPUT_DIR / "ner_pos_features.csv", index=False)
ner_token_feature_assignment.to_csv(OUTPUT_DIR / "ner_token_feature_assignment.csv", index=False)
contextual_if_then_rules_table.to_csv(OUTPUT_DIR / "ner_contextual_if_then_rules.csv", index=False)
morphological_if_then_rules_table.to_csv(OUTPUT_DIR / "ner_morphological_if_then_rules.csv", index=False)
pos_if_then_rules_table.to_csv(OUTPUT_DIR / "ner_pos_if_then_rules.csv", index=False)
ner_if_then_rules_table.to_csv(OUTPUT_DIR / "ner_if_then_rules.csv", index=False)

print("Table 1. List of contextual features")
display(contextual_feature_table)
display(HTMLBlock(render_if_then_rules_html(contextual_if_then_rules_table)))
print("Table 2. List of morphological features")
display(morphological_feature_table)
display(HTMLBlock(render_if_then_rules_html(morphological_if_then_rules_table)))
print("Table 3. List of part-of-speech features")
display(pos_feature_table)
display(HTMLBlock(render_if_then_rules_html(pos_if_then_rules_table)))
print("Table 4. Result of tokenization and feature assignment processes")
print("Komentar contoh:", sample_text_for_ner_steps)
display(ner_token_feature_assignment.head(80))
display(HTMLBlock(render_if_then_rules_html(ner_if_then_rules_table)))


Table 1. List of contextual features


,Feature Name,Explanation,Example
0,PPRE,Person prefix,"Dr., Pak, Bpk, K.H."
1,PMID,Person middle,"bin, van"
2,PSUF,Person suffix,"S.Kom, SH"
3,PTIT,Person title,"Menteri, Presiden, Pak"
4,OPRE,Organization prefix,"PT, Universitas, Kementerian"
5,OSUF,Organization suffix,"Ltd., Company, Tbk"
6,OPOS,Position in organization,"Ketua, Menteri, Rektor"
7,OCON,Other organization contextual,"Mukernas, Rakernas, Gojek"
8,LPRE,Location prefix,"Kota, Provinsi, Kabupaten"
9,LSUF,Location suffix,"Utara, Selatan, City"


HTML visualization generated.

Table 2. List of morphological features


,Feature Name,Explanation,Example
0,TitleCase,Begin with uppercase letter and followed by lowercase letter,Soedirman
1,UpperCase,All uppercase letter,KPU
2,LowerCase,All lowercase letter,menuntut
3,MixedCase,Uppercase and lowercase letter,LeIP
4,CapStart,Begin with uppercase letter,"LeIP, Muhammad"
5,CharDigit,Letter and number,P3K
6,Digit,All number,2004
7,DigitSlash,Number with slash,17/5
8,Numeric,Number with dot or comma,"20,5; 17.500,00"
9,NumStr,Number in word,"satu, tujuh, lima"


...

,Feature Name,Explanation,Example
0,ART,Article,"si, sang"
1,ADJ,Adjective,"indah, baik, benar"
2,ADV,Adverb,"telah, kemarin, sudah"
3,AUX,Auxiliary verb,"harus, bisa"
4,C,Conjunction,"dan, atau, lalu"
5,DEF,Definition,"merupakan, adalah"
6,NOUN,Noun,"rumah, hukum, korupsi"
7,NOUNP,Personal noun,"ayah, ibu, pak"
8,NUM,Number,"satu, dua, 500"
9,MODAL,Modal,"akan, mau"


HTML visualization generated.

Table 4. Result of tokenization and feature assignment processes
Komentar contoh: Kasus nya makarim apa ? 

INDOSURYA YANG MERAMPOK UANG KORBAN 16,3 triliun dan serara keseluruhan dari 23 ribu nasabahnya total 106 triliun. 

Tapj Mahkamah Agung menjatuhkan vonis 18 tahun penjara dan denda hanya Rp15 miliar kepada pimpinan Koperasi Simpan Pinjam (KSP) Indosurya, Henry Surya, yang diduga merugikan 23 ribu orang, dengan total kerugian disebut mencapai Rp106 triliun


Dan saat ini uang dan aset yg masih ada blm jelas kapan akan di kembalikan kepada para korban.


,Token string,Token kind,Contextual features,Morphological features,Part-of-speech features
0,Kasus,WORD,,"TitleCase, CapStart",OOV
1,nya,WORD,,LowerCase,OOV
2,makarim,WORD,,LowerCase,OOV
3,apa,WORD,,LowerCase,OOV
4,?,OPUNC,,,
...,...,...,...,...,...
75,jelas,WORD,,LowerCase,OOV
76,kapan,WORD,,LowerCase,OOV
77,akan,WORD,,LowerCase,MODAL
78,di,WORD,LOPP,LowerCase,PREP


HTML visualization generated.

In [16]:
texts_for_ner = result_df[TEXT_COL].tolist()
comment_ids_for_ner = result_df["comment_id"].tolist()
NER_OUTPUT_PATH = OUTPUT_DIR / f"ner_entities_rule_based_{DATA_PATH.stem}.csv"

ner_df = run_rule_based_ner(texts_for_ner, comment_ids_for_ner)
ner_df.to_csv(NER_OUTPUT_PATH, index=False)
ner_df.to_csv(OUTPUT_DIR / "ner_entities.csv", index=False)

print("Rule-based NER selesai.")
print("Jumlah entitas:", len(ner_df))
print("Output:", NER_OUTPUT_PATH)
ner_df.head(20)


Rule-based NER selesai.
Jumlah entitas: 568
Output: outputs\ner_entities_rule_based_komentar_youtube.csv


,doc_id,comment_id,text,entity,label,rule_name,score,start,end
0,1,2,Perlawanan ojol dgn tolak jokodok,jokodok,PER,person_jokowi,1.0,26,33
1,1,2,Perlawanan ojol dgn tolak jokodok,ojol,ORG,org_gojek,1.0,11,15
2,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",Nadiem,PER,person_nadiem,1.0,132,138
3,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",Indonesia,GPE,geo_indonesia,1.0,49,58
4,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",Indonesia,GPE,geo_indonesia,1.0,104,113
5,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",Pilpres,EVT,event_terms,1.0,0,7
6,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",280 juta,MON,money_terms,1.0,34,42
7,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",1 orang,QTY,quantity_terms,1.0,96,103
8,5,6,"Yg membuka lapangan kerja ber juta orang di Indonesia di bui,sedangkan koruptor di Indonesia yg merajalela,semakin solid,(pasti bung Karno&Soeharto)sedih me...",Indonesia,GPE,geo_indonesia,1.0,44,53
9,5,6,"Yg membuka lapangan kerja ber juta orang di Indonesia di bui,sedangkan koruptor di Indonesia yg merajalela,semakin solid,(pasti bung Karno&Soeharto)sedih me...",Indonesia,GPE,geo_indonesia,1.0,83,92


## 6. Tabel Hasil Akhir

Bagian ini merapikan hasil utama LDA, clustering FTC, dan rule-based NER menjadi tabel yang siap dibaca atau dimasukkan ke laporan.


In [17]:
lda_result_table = (
    topic_words.sort_values(["topic_id", "rank"])
    .groupby("topic_id")
    .agg(
        top_terms=("term", lambda terms: ", ".join(list(terms)[:10])),
        top_probability=("probability", "max"),
    )
    .reset_index()
)

topic_counts = (
    result_df["topic_id"]
    .value_counts()
    .sort_index()
    .rename_axis("topic_id")
    .reset_index(name="n_comments")
)

topic_confidence = (
    result_df.groupby("topic_id")["topic_confidence"]
    .mean()
    .round(4)
    .reset_index(name="avg_topic_confidence")
)

lda_result_table = (
    lda_result_table
    .merge(topic_counts, on="topic_id", how="left")
    .merge(topic_confidence, on="topic_id", how="left")
)

lda_result_table[["topic_id", "n_comments", "avg_topic_confidence", "top_terms", "top_probability"]]


,topic_id,n_comments,avg_topic_confidence,top_terms,top_probability
0,0,127,0.6435,"pak, allah, nadiem, adil, moga, siapa, hakim, tuhan, orang, bapak",0.057076
1,1,99,0.6309,"orang, sama, baik, nadim, jangan, punya, negara, korupsi, maling, lah",0.058116
2,2,87,0.6458,"jokowi, korupsi, apa, uang, nadiem, semua, sebut, vonis, pak, koruptor",0.127570
3,3,88,0.7213,"hukum, indonesia, negara, gila, rakyat, orang, jahat, buat, adil, koruptor",0.083700
4,4,99,0.6965,"jadi, tumbal, mau, nadiem, jokowi, tri, menteri, salah, pak, masuk",0.116125


In [18]:
cluster_result_table = cluster_summary[[
    "cluster_id",
    "n_comments",
    "ftc_termset",
    "entropy_overlap",
    "dominant_topic",
    "dominant_topic_terms",
    "cluster_top_terms",
    "example_comment",
]].copy()

cluster_result_table


,cluster_id,n_comments,ftc_termset,entropy_overlap,dominant_topic,dominant_topic_terms,cluster_top_terms,example_comment
0,0,8,"hukum, jadi",1.590554,4,"jadi, tumbal, mau, nadiem, jokowi","jadi, hukum, nadiem, orang, jokowi, hakim, tumbal, tuhan, uda, negara","Hukum sudah menjadi alat kekuasaan,\nIni Uda menjadi negara tiran.\nSudah wajib hukum nya dilakukan revolusi akhlak\nAgar GK menjadi makin banyak makan korb..."
1,1,8,lalu,1.548112,1,"orang, sama, baik, nadim, jangan","lalu, aku, peran, engkau, tubuh, jasmani, orang, panggung, diri, akal",Lalu bagaimana kabar Dadang sekarang
2,2,8,"orang, pak",1.572495,0,"pak, allah, nadiem, adil, moga","pak, orang, bapak, tak, tuhan, jadi, siapa, baik, nadiem, mungkin","Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya"
3,3,8,didik,1.859656,3,"hukum, indonesia, negara, gila, rakyat","didik, jadi, indonesia, kamu, nadiem, moga, mas, besar, lebih, mundur","Begitu besar jasa Nadiem untuk Pendidikan Indonesia...itu jelas , tidak bisa dipungkiri... semoga hukum bicara lebih baik lebih adil.\nNadiem... semoga Alla..."
4,4,8,"jadi, nadiem",1.927286,4,"jadi, tumbal, mau, nadiem, jokowi","nadiem, jadi, jokowi, menteri, tumbal, pak, salah, tahu, astagfirulloh, uang",Kasihan Nadiem..dia korban dari Jokowi..kesalahan Nadiem kenapa dia mau menjadi menteri..
5,5,8,ngeri,2.030486,3,"hukum, indonesia, negara, gila, rakyat","ngeri, indonesia, birokrasi, nadiem, gila, hukum, bro, nepalisasi, konoha, cepat",Ngeri dengan hukum di indonesia
6,6,8,jujur,2.125935,4,"jadi, tumbal, mau, nadiem, jokowi","jujur, orang, jadi, pak, nadiem, masuk, baik, biangkeroknya, tangkap, siapa",Biangkeroknya tangkap! Tapi siapa yang bisa nangkap???😢 semoga Tuhan memberikan hidajah untuk orang2 jujur. Aamiin
7,7,8,"jabat, jadi",1.992404,4,"jadi, tumbal, mau, nadiem, jokowi","jadi, jabat, mau, menteri, negeri, tumbal, anak, jokowi, kaya, orang",Ngapain jadi pejabat mending tolak klw ujung nya menyakitkan
8,8,8,konoha,2.010855,3,"hukum, indonesia, negara, gila, rakyat","konoha, politik, mau, negara, beliau, memang, paling, aneh, dunia, nadiem",Negara Konoha memang negara paling aneh sedunia..
9,9,9,lah,2.106305,1,"orang, sama, baik, nadim, jangan","lah, nadim, gojek, sama, orang, amin, allah, buka, ngapain, jadi",Ya bongkar lah semuaa\nMasa sudah tersudut masih belum dibuka😅


In [19]:
ner_columns = ["doc_id", "comment_id", "text", "entity", "label", "rule_name", "score", "start", "end"]
NER_OUTPUT_PATH = OUTPUT_DIR / f"ner_entities_rule_based_{DATA_PATH.stem}.csv"

if "ner_df" in globals() and isinstance(ner_df, pd.DataFrame) and not ner_df.empty:
    ner_result_table = ner_df.copy()
elif NER_OUTPUT_PATH.exists():
    ner_result_table = pd.read_csv(NER_OUTPUT_PATH)
else:
    ner_result_table = pd.DataFrame(columns=ner_columns)

available_ner_columns = [column for column in ner_columns if column in ner_result_table.columns]
ner_result_table = ner_result_table[available_ner_columns].copy()

if "score" in ner_result_table.columns and not ner_result_table.empty:
    ner_result_table["score"] = ner_result_table["score"].astype(float).round(4)

ner_result_table.head(50)


,doc_id,comment_id,text,entity,label,rule_name,score,start,end
0,1,2,Perlawanan ojol dgn tolak jokodok,jokodok,PER,person_jokowi,1.0,26,33
1,1,2,Perlawanan ojol dgn tolak jokodok,ojol,ORG,org_gojek,1.0,11,15
2,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",Nadiem,PER,person_nadiem,1.0,132,138
3,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",Indonesia,GPE,geo_indonesia,1.0,49,58
4,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",Indonesia,GPE,geo_indonesia,1.0,104,113
5,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",Pilpres,EVT,event_terms,1.0,0,7
6,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",280 juta,MON,money_terms,1.0,34,42
7,2,3,"Pilpres yg untuk menentukan nasib 280 juta orang Indonesia saja bisa diatur, apalagi cuma untuk 1 orang Indonesia ... ?! Ya gak pak Nadiem ?! Sabar lah ya",1 orang,QTY,quantity_terms,1.0,96,103
8,5,6,"Yg membuka lapangan kerja ber juta orang di Indonesia di bui,sedangkan koruptor di Indonesia yg merajalela,semakin solid,(pasti bung Karno&Soeharto)sedih me...",Indonesia,GPE,geo_indonesia,1.0,44,53
9,5,6,"Yg membuka lapangan kerja ber juta orang di Indonesia di bui,sedangkan koruptor di Indonesia yg merajalela,semakin solid,(pasti bung Karno&Soeharto)sedih me...",Indonesia,GPE,geo_indonesia,1.0,83,92
